[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C2/notebooks/praporek.ipynb)

# Červený praporek: vak, kterému se nedá věřit

Notebook k bloku 6 druhého cvičení (15 min). **Nic se neodevzdává** — na konci
se řekne nahlas, jak jste na to přišli.

> Skript níž **běží**. Hlásí `status = optimal`, vypíše kompletní složení vaku
> parenterální výživy, jeho cenu — a jeho vlastní **kontrola
> dosazením zpět projde 6 ze 6 omezení**.
>
> Přesto se tomu výsledku nedá věřit.
>
> **Doložte číslem, co je na něm špatně.** Ne „zdá se mi to divné", ale
> konkrétní hodnota, která porušuje zadání.

Úloha je táž jako v notebooku [`matice.ipynb`](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C2/notebooks/matice.ipynb): šest přípravků,
čtyři živiny, dvoulitrový vak. Celý model je přímo tady v notebooku, takže se
v něm dá hledat i experimentovat.

## Postup

1. spustit model a přečíst si, co vypsal;
2. spustit jeho kontrolu dosazením zpět — projde;
3. porovnat výsledek se **seznamem požadavků ze zadání** *(díra 1)*;
4. doplnit, co v modelu chybí, a spočítat, co ta chyba stojí *(díra 2)*;
5. napsat jednu větu: **které omezení chybí a o kolik se to liší**.

Kdo bude hotov dřív, dostane druhou otázku: *jak by se taková chyba dala
odhalit automaticky?*

---

*Čísla, se kterými se notebook porovnává, počítají skripty [`kod/cerveny_praporek_tuky.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C2/kod/cerveny_praporek_tuky.py), [`kod/infuze_matice.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C2/kod/infuze_matice.py). Notebook sám staví [`kod/postav_notebooky.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C2/kod/postav_notebooky.py) — ruční úprava `.ipynb` se při dalším spuštění přepíše.*

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp

In [ ]:
import cvxpy as cp
import numpy as np

## Model, který dostáváte

Data jsou stejná jako v maticovém rozšíření: šest přípravků (sloupce) a čtyři
živiny (řádky matice $A$).

In [ ]:
NAZVY = ["Glc 10%", "Glc 40%", "AK 10%", "Lipid 20%", "KCl 7,45%", "voda"]
CENA = np.array([40.0, 120.0, 350.0, 500.0, 30.0, 15.0])      # Kč za litr

# řádky: energie [kcal/l], bílkoviny [g/l], draslík [mmol/l], objem [l/l]
A = np.array([
    [400.0, 1600.0, 400.0, 2000.0,    0.0, 0.0],
    [  0.0,    0.0, 100.0,    0.0,    0.0, 0.0],
    [  0.0,    0.0,   0.0,    0.0, 1000.0, 0.0],
    [  1.0,    1.0,   1.0,    1.0,    1.0, 1.0],
])
ENERGIE, BILKOVINY, DRASLIK_MIN, DRASLIK_MAX, OBJEM = 1800.0, 70.0, 60.0, 100.0, 2.0
LIPID_MAX = 0.5      # l, technologický strop tukové emulze

### Sestavení a řešení

`sestav()` vrátí proměnnou a slovník omezení, `vyres()` z toho udělá úlohu
a vyřeší ji. Parametr `pridej` se hodí až v posledních buňkách.

In [ ]:
def sestav():
    """Vrátí proměnnou a slovník omezení {popis: výraz}."""
    x = cp.Variable(6, nonneg=True)
    omezeni = {
        "energie >= 1800 kcal": A[0] @ x >= ENERGIE,
        "bilkoviny >= 70 g": A[1] @ x >= BILKOVINY,
        "draslik >= 60 mmol": A[2] @ x >= DRASLIK_MIN,
        "draslik <= 100 mmol": A[2] @ x <= DRASLIK_MAX,
        "objem == 2 l": A[3] @ x == OBJEM,
        "lipid <= 0,5 l": x[3] <= LIPID_MAX,
    }
    return x, omezeni


def vyres(pridej=None):
    """Vyřeší úlohu. `pridej` je funkce, která z x udělá slovník DALŠÍCH omezení."""
    x, omezeni = sestav()
    if pridej is not None:
        omezeni.update(pridej(x))
    uloha = cp.Problem(cp.Minimize(CENA @ x), list(omezeni.values()))
    uloha.solve(solver=cp.CLARABEL, tol_gap_abs=1e-11, tol_gap_rel=1e-11, tol_feas=1e-11)
    duals = {k: float(np.atleast_1d(v.dual_value)[0]) for k, v in omezeni.items()}
    return np.asarray(x.value).ravel(), float(uloha.value), uloha.status, duals, omezeni


x_opt, cena_opt, status, duals, omezeni = vyres()

print(f"status = {status}")
print("\nsložení vaku:")
for nazev, objem in zip(NAZVY, x_opt):
    print(f"  {nazev:10s} {objem:7.4f} l")
print(f"\ncena = {cena_opt:.2f} Kč za vak")

## Kontrola dosazením zpět — a projde

Tahle buňka vypadá jako poctivá kontrola, a formálně poctivá kontrola je: projde
každé omezení modelu a dosadí do něj nalezené řešení. Výsledek je **6 ze 6**.

Až budete hledat vadu, vraťte se sem a rozmyslete si, co přesně tahle kontrola
kontroluje.

In [ ]:
TOL = 1e-6
proslo = 0
for nazev, vyraz in omezeni.items():
    zbytek = float(np.atleast_1d(vyraz.violation()).max())
    ok = zbytek <= TOL
    proslo += ok
    print(f"  {nazev:22s} odchylka {zbytek:9.2e}  {'OK' if ok else 'PORUŠENO'}")
print(f"\nkontrola dosazením zpět: {proslo}/{len(omezeni)} omezení splněno")

## Úkol: seznam požadavků ze zadání

Toto je **úplné zadání výukového modelu**. Porovnejte ho s výsledkem a kódem.

| požadavek | hodnota |
|---|---|
| energie | aspoň 1 800 kcal |
| bílkoviny | aspoň 70 g |
| draslík | 60 až 100 mmol |
| objem vaku | přesně 2 l |
| tuková emulze | **aspoň 0,25 l** a nejvýš 0,5 l |

**Díra 1:** napište kontrolu, která porovná nalezené řešení s tímhle seznamem —
ne s omezeními modelu. Na rozdíl od buňky výš **má spadnout**, a hláška assertu
má obsahovat naměřené číslo. To je ten „doklad", o který v celém bloku jde.

In [ ]:
# TODO 1: napište kontrolu proti seznamu požadavků ZE ZADÁNÍ (tabulka výš),
# ne proti omezením modelu. Jeden z požadavků v modelu chybí -- a právě
# na něm to má spadnout. Do hlášky assertu dejte i naměřené číslo.
def kontrola_proti_zadani(x):
    ...   # asserty: energie, bílkoviny, draslík, objem, tuky


kontrola_proti_zadani(x_opt)
print("kontrola proti zadání prošla — opravdu?")

## Co ta chyba stojí

Teď je vidět, které omezení v modelu chybí. Doplňte ho a spočítejte rozdíl —
teprve tím je vada **doložená**: ne „chybí tam tuky", ale „chybí tam tuky a je
to o tolik a tolik korun".

In [ ]:
# TODO 2: doplňte omezení, které v modelu chybí, a spočítejte, co stojí.
def chybejici(x):
    return {"popis": ...}   # {popis omezení: výraz s proměnnou x}


x_ok, cena_ok, _, _, _ = vyres(pridej=chybejici)

print(f"model ze skriptu   lipid {x_opt[3]:.3f} l   {cena_opt:.2f} Kč")
print(f"s doplněným tukem  lipid {x_ok[3]:.3f} l   {cena_ok:.2f} Kč")
print(f"rozdíl: {cena_ok - cena_opt:.2f} Kč — o tolik je rozbitý model levnější")

assert x_opt[3] < 1e-6, "rozbitý model měl vyjít úplně bez lipidů"
assert abs(x_ok[3] - 0.25) < 1e-6, "doplněné omezení má být aktivní (0,25 l)"
assert abs(cena_ok - 453.00) < 0.01, "cena s tuky se rozešla s číslem ze skriptu"
assert abs((cena_ok - cena_opt) - 88.33) < 0.01, "rozdíl se rozešel s číslem ze skriptu"
print("\nlevnější výsledek nesplňuje zadání")

### Vaše věta

*(dvojklikem do téhle buňky se do ní dá psát; jedna věta, s číslem)*

**TODO:** V modelu chybí omezení … a stojí to …

## Poučení

Kontrola omezení z kódu neodhalila požadavek, který v něm chyběl.
Proto výsledek porovnáváme s **úplným zadáním**.